In [1]:
import rasterio
import os

def inspect_sar_file(tiff_path):
    """Extracts structural and geospatial profiles of the raw SAR image."""
    if not os.path.exists(tiff_path):
        raise FileNotFoundError(f"Could not find the GeoTIFF at: {tiff_path}")
        
    with rasterio.open(tiff_path) as src:
        profile = src.profile
        print(f"--- Metadata Profile for {os.path.basename(tiff_path)} ---")
        print(f"Data Type:        {profile['dtype']}")
        print(f"Dimensions:       {profile['width']}x{profile['height']} pixels")
        print(f"Bands:            {profile['count']}")
        print(f"CRS (Projection): {profile['crs']}")
        print(f"NoData Value:     {profile['nodata']}")
        print(f"Bounds:           {src.bounds}\n")
        return profile

# Test this on Month 1 VV and Month 2 VV
m1_vv_path = "data/S1A_IW_GRDH_1SDV_20260514T010316_20260514T010341_064506_081FE3_B4B1_COG.SAFE/measurement/s1a-iw-grd-vv-20260514t010316-20260514t010341-064506-081fe3-001-cog.tiff"
m2_vv_path = "data/S1A_IW_GRDH_1SDV_20260619T010315_20260619T010340_065031_083245_3C65.SAFE/measurement/s1a-iw-grd-vv-20260619t010315-20260619t010340-065031-083245-001.tiff"

profile_m1 = inspect_sar_file(m1_vv_path)
profile_m2 = inspect_sar_file(m2_vv_path)

# Verify structural alignment
if profile_m1['crs'] == profile_m2['crs'] and profile_m1['width'] == profile_m2['width']:
    print("Success: Spatial structures match.")
else:
    print("Warning: Dimensions or CRS mismatch. Need to crop or reproject.")


--- Metadata Profile for s1a-iw-grd-vv-20260514t010316-20260514t010341-064506-081fe3-001-cog.tiff ---
Data Type:        uint16
Dimensions:       25530x16734 pixels
Bands:            1
CRS (Projection): None
NoData Value:     0.0
Bounds:           BoundingBox(left=0.0, bottom=16734.0, right=25530.0, top=0.0)

--- Metadata Profile for s1a-iw-grd-vv-20260619t010315-20260619t010340-065031-083245-001.tiff ---
Data Type:        uint16
Dimensions:       25530x16733 pixels
Bands:            1
CRS (Projection): None
NoData Value:     None
Bounds:           BoundingBox(left=0.0, bottom=16733.0, right=25530.0, top=0.0)

Success: Spatial structures match.


In [2]:
import numpy as np
import rasterio
from rasterio.windows import Window

def generate_aligned_patches(path_t1, path_t2, patch_size=128):
    """
    Safely slides windows across two slightly misaligned raw SAR scenes,
    converts raw uint16 DN values to Decibels (dB), and scales to.
    """
    with rasterio.open(path_t1) as src1, rasterio.open(path_t2) as src2:
        # Resolve the dimension mismatch by taking the absolute minimum intersection
        max_h = min(src1.height, src2.height)
        max_w = min(src1.width, src2.width)
        
        print(f"Streaming patches using intersection bounding box: {max_w}x{max_h}")
        
        for y in range(0, max_h, patch_size):
            for x in range(0, max_w, patch_size):
                # Ensure the window does not exceed the intersection boundary
                w_width = min(patch_size, max_w - x)
                w_height = min(patch_size, max_h - y)
                window = Window(x, y, w_width, w_height)
                
                # Read identical window positions
                tile1 = src1.read(1, window=window).astype(np.float32)
                tile2 = src2.read(1, window=window).astype(np.float32)
                
                # Math: Convert Amplitude DN values to Decibels (dB)
                tile1_db = 20 * np.log10(np.abs(tile1) + 1e-6)
                tile2_db = 20 * np.log10(np.abs(tile2) + 1e-6)
                
                # Normalize: Map expected radar ranges (-30dB to 0dB) into [0.0, 1.0]
                tile1_norm = np.clip((tile1_db - (-30)) / (0 - (-30)), 0.0, 1.0)
                tile2_norm = np.clip((tile2_db - (-30)) / (0 - (-30)), 0.0, 1.0)
                
                # Quality Gate: Ignore empty black border data gaps (common in raw SAR edges)
                if np.mean(tile1) < 1.0 or np.mean(tile2) < 1.0:
                    continue
                    
                yield tile1_norm, tile2_norm, window


In [3]:
def compute_baseline_anomalies(path_t1, path_t2, patch_size=128):
    """
    Generates a localized non-ML change detection baseline using a Log-Ratio approach.
    """
    # Initialize a downscaled list or dictionary to preview results without memory exhaustion
    sample_anomalies = []
    
    patch_generator = generate_aligned_patches(path_t1, path_t2, patch_size)
    
    for t1_patch, t2_patch, window in patch_generator:
        # Log-Ratio calculation: Adding small epsilon avoids division-by-zero
        # Mathematically equivalent to log(t2) - log(t1) on unnormalized data
        ratio = (t2_patch + 1e-5) / (t1_patch + 1e-5)
        log_ratio = np.log(ratio)
        
        # Calculate localized anomaly detection thresholding via standard deviations
        mean_val = np.mean(log_ratio)
        std_val = np.std(log_ratio)
        
        # Flag pixels deviating by more than 2.5 standard deviations as anomalies
        anomaly_mask = np.abs(log_ratio - mean_val) > (2.5 * std_val)
        
        # If an anomaly is detected inside the patch, track it for debugging
        if np.any(anomaly_mask):
            sample_anomalies.append({
                "window": (window.col_off, window.row_off),
                "anomaly_ratio": np.sum(anomaly_mask) / (patch_size ** 2)
            })
            
    print(f"Processed baseline. Found {len(sample_anomalies)} patches with significant change anomalies.")
    return sample_anomalies


In [4]:
%pip install torch torchvision torchaudio --no-index --find-links ~/pip_cache

Looking in links: /home/aryakumar/pip_cache
Note: you may need to restart the kernel to use updated packages.


In [5]:
import torch
import numpy as np
from torch.utils.data import IterableDataset


class SARSpatialDataset(IterableDataset):
    def __init__(self, path_t1, path_t2, patch_size=128):
        super().__init__()

        self.path_t1 = path_t1
        self.path_t2 = path_t2
        self.patch_size = patch_size

    def _pad_patch(self, patch):
        patch = np.asarray(patch)

        # Safety check
        if patch.ndim != 2:
            raise ValueError(
                f"Expected a 2D SAR patch, got shape {patch.shape}"
            )

        h, w = patch.shape
        target = self.patch_size

        # Already correct size
        if h == target and w == target:
            return patch

        # Prevent oversized patches from silently being cropped
        if h > target or w > target:
            raise ValueError(
                f"Patch is larger than patch_size={target}: "
                f"got shape {patch.shape}"
            )

        # Create padded patch
        padded = np.zeros(
            (target, target),
            dtype=patch.dtype
        )

        padded[:h, :w] = patch

        return padded

    def __iter__(self):

        patch_generator = generate_aligned_patches(
            self.path_t1,
            self.path_t2,
            self.patch_size
        )

        for tile1_norm, tile2_norm, window in patch_generator:
            tile1_norm = self._pad_patch(tile1_norm)
            tile2_norm = self._pad_patch(tile2_norm)

            x1 = torch.from_numpy(
                np.asarray(tile1_norm, dtype=np.float32)
            ).unsqueeze(0)

            x2 = torch.from_numpy(
                np.asarray(tile2_norm, dtype=np.float32)
            ).unsqueeze(0)

            metadata = torch.tensor(
                [
                    window.col_off,
                    window.row_off,
                    window.width,
                    window.height
                ],
                dtype=torch.int64
            )

            yield x1, x2, metadata


In [6]:
import torch
import torch.nn as nn

class SiameseSARNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Shared deep feature extraction pipeline
        self.feature_extractor = nn.Sequential(
            # Layer 1: Expand channels to map local variations
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            # Layer 2: Capture medium-range context
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            
            # Layer 3: Consolidate structural descriptors
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            # Layer 4: Final latent feature mapping space
            nn.Conv2d(32, 16, kernel_size=3, padding=1),
            nn.ReLU()
        )

    def forward_once(self, x):
        """Passes a single timestamp patch through the feature extractor."""
        return self.feature_extractor(x)

    def forward(self, x1, x2):
        """
        Passes both inputs through the shared weights and computes 
        the Euclidean distance across feature channels.
        """
        # Feature shape: [Batch, 16 Channels, Height, Width]
        feat_t1 = self.forward_once(x1)
        feat_t2 = self.forward_once(x2)
        
        # Calculate pixel-wise Euclidean Distance: sqrt( sum( (f1 - f2)^2 ) )
        # Dim=1 sums across the channel axis, preserving spatial Height and Width
        distance_map = torch.sqrt(torch.sum((feat_t1 - feat_t2) ** 2, dim=1))
        
        return distance_map


In [7]:
from torch.utils.data import DataLoader
# Assuming files are named model.py and dataset.py
# from dataset import SARSpatialDataset
# from model import SiameseSARNet

def run_ai_anomaly_inference(m1_path, m2_path, patch_size=128):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Executing pipeline on device: {device}")
    
    # 1. Initialize dataset and dataloader
    dataset = SARSpatialDataset(m1_path, m2_path, patch_size=patch_size)
    dataloader = DataLoader(dataset, batch_size=4) # Process 4 patch pairs at a time
    
    # 2. Instantiate Model
    model = SiameseSARNet().to(device)
    model.eval() # Set to evaluation mode
    
    all_anomaly_patches = []
    
    with torch.no_grad():
        for batch_idx, (x1, x2, metadata) in enumerate(dataloader):
            x1, x2 = x1.to(device), x2.to(device)
            
            # Generate the feature distance maps from the Siamese model
            # Output shape: [Batch, Height, Width]
            distance_maps = model(x1, x2)
            
            # Convert tensors back to numpy to stage for Phase 5 integration
            distance_maps_np = distance_maps.cpu().numpy()
            metadata_np = metadata.numpy()
            
            for i in range(distance_maps_np.shape[0]):
                all_anomaly_patches.append({
                    "distance_map": distance_maps_np[i],
                    "col_off": metadata_np[i][0],
                    "row_off": metadata_np[i][1],
                    "width": metadata_np[i][2],
                    "height": metadata_np[i][3]
                })
                
    print(f"AI Inference complete. Generated features for {len(all_anomaly_patches)} tiles.")
    return all_anomaly_patches


In [8]:
import numpy as np

# Define File Paths
m1_vv_path = "data/S1A_IW_GRDH_1SDV_20260514T010316_20260514T010341_064506_081FE3_B4B1_COG.SAFE/measurement/s1a-iw-grd-vv-20260514t010316-20260514t010341-064506-081fe3-001-cog.tiff"
m2_vv_path = "data/S1A_IW_GRDH_1SDV_20260619T010315_20260619T010340_065031_083245_3C65.SAFE/measurement/s1a-iw-grd-vv-20260619t010315-20260619t010340-065031-083245-001.tiff"

# Analytical Baseline
print(">>> Running Phase 3: Computing Non-ML Analytical Log-Ratio Baseline...")
baseline_anomalies = compute_baseline_anomalies(m1_vv_path, m2_vv_path, patch_size=128)

# AI Feature Distance Extraction
print("\n>>> Running Phase 4: Streaming Patches Through Siamese Network...")
# Note: Fixed a minor index parsing typo from the prior structural layout snippet
ai_results = run_ai_anomaly_inference(m1_vv_path, m2_vv_path, patch_size=128)

# Quality Control Check
print("\n--- Pipeline Execution Summary ---")
print(f"Total tiles evaluated by baseline engine: {len(baseline_anomalies)}")
print(f"Total feature maps generated by Siamese AI: {len(ai_results)}")

if len(ai_results) > 0:
    print(f"Sample Output Array Shape: {ai_results[0]['distance_map'].shape} (Expects 128x128)")


>>> Running Phase 3: Computing Non-ML Analytical Log-Ratio Baseline...
Streaming patches using intersection bounding box: 25530x16733
Processed baseline. Found 128 patches with significant change anomalies.

>>> Running Phase 4: Streaming Patches Through Siamese Network...
Executing pipeline on device: cuda
Streaming patches using intersection bounding box: 25530x16733
AI Inference complete. Generated features for 25807 tiles.

--- Pipeline Execution Summary ---
Total tiles evaluated by baseline engine: 128
Total feature maps generated by Siamese AI: 25807
Sample Output Array Shape: (128, 128) (Expects 128x128)


In [9]:
import numpy as np

def reconstruct_global_anomaly_map(ai_results, global_width=25530, global_height=16733):
    print(f"Allocating global canvas matrix: {global_width}x{global_height} pixels...")
    # Initialize an empty canvas for continuous distance values
    distance_canvas = np.zeros((global_height, global_width), dtype=np.float32)
    
    # Stamp each 128x128 patch back using its recorded pixel offsets
    for patch in ai_results:
        x, y = patch["col_off"], patch["row_off"]
        w, h = patch["width"], patch["height"]
        
        # Extract the 128x128 distance tile
        tile = patch["distance_map"]
        
        # Map it safely to the global grid (slicing handles edge-boundary truncation)
        distance_canvas[y:y+h, x:x+w] = tile[:h, :w]
        
    print("Canvas assembly complete. Calculating global statistical threshold...")
    
    # Isolate anomalies using statistical standard deviation (Unsupervised strategy)
    # Extract metrics only from non-zero elements to avoid bias from background empty spaces
    valid_pixels = distance_canvas[distance_canvas > 0]
    global_mean = np.mean(valid_pixels) if len(valid_pixels) > 0 else 0
    global_std = np.std(valid_pixels) if len(valid_pixels) > 0 else 1
    
    # Flag pixels that deviate significantly from standard structural variations
    # Adjust the multiplier (e.g., 2.5 or 3.0) to fine-tune sensitivity
    anomaly_threshold = global_mean + (2.5 * global_std)
    binary_anomaly_mask = distance_canvas > anomaly_threshold
    
    print(f"Threshold set at {anomaly_threshold:.4f}. Found {np.sum(binary_anomaly_mask)} anomaly pixels.")
    return binary_anomaly_mask, distance_canvas

# Run the reconstruction
anomaly_mask, raw_distance_grid = reconstruct_global_anomaly_map(ai_results)


Allocating global canvas matrix: 25530x16733 pixels...
Canvas assembly complete. Calculating global statistical threshold...
Threshold set at 0.0577. Found 1365 anomaly pixels.


In [10]:
%pip install scipy

Note: you may need to restart the kernel to use updated packages.


In [11]:
import json
from scipy.ndimage import label

def export_pixel_anomalies_to_json(binary_mask, min_cluster_size=10, output_path="anomalies.json"):
    """
    Groups individual anomaly pixels into contiguous clusters, filters out single-pixel 
    radar noise, and exports their bounding box coordinates.
    """
    print("Grouping individual pixels into spatial anomaly clusters...")
    # Label distinct interconnected structural islands of anomalies
    labeled_mask, num_features = label(binary_mask)
    
    anomaly_vectors = []
    
    for i in range(1, num_features + 1):
        # Find index coordinates of the current cluster
        pixel_indices = np.argwhere(labeled_mask == i)
        cluster_size = len(pixel_indices)
        
        # Filter out minor speckle noise spikes
        if cluster_size < min_cluster_size:
            continue
            
        # Extract pixel boundary boxes: [Min Row, Min Col, Max Row, Max Col]
        min_y, min_x = pixel_indices.min(axis=0)
        max_y, max_x = pixel_indices.max(axis=0)
        
        anomaly_vectors.append({
            "id": i,
            "pixel_area_size": int(cluster_size),
            "pixel_bounding_box": {
                "col_start": int(min_x),
                "row_start": int(min_y),
                "col_end": int(max_x),
                "row_end": int(max_y)
            }
        })
        
    # Write to local file storage
    with open(output_path, "w") as f:
        json.dump(anomaly_vectors, f, indent=4)
        
    print(f"Export complete. Saved {len(anomaly_vectors)} distinct verified structural anomalies to {output_path}.")
    return anomaly_vectors

# Run the export script
extracted_vectors = export_pixel_anomalies_to_json(anomaly_mask)


Grouping individual pixels into spatial anomaly clusters...
Export complete. Saved 16 distinct verified structural anomalies to anomalies.json.
